# Pipeline comparison statistics

This notebook compares the **envelope-only** pipeline with the **envelope + onsets** pipeline.

It performs two mixed-model analyses:

1. **Decoding accuracy**: logistic mixed model  
   `correct ~ pipeline + group + (1 | subject)`

2. **Correlation separation**: linear mixed model on Fisher-z transformed attended–ignored difference  
   `delta_z ~ pipeline + group + (1 | subject)`

The main effect of interest is **pipeline**. The secondary fixed effect is **hearing group**. Subject is included as a random intercept.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# pymer4 is used because it gives lme4-style mixed models from Python.
# If this import fails, run this notebook in the same environment where your GLMM scripts work.
try:
    from pymer4.models import glmer, lmer
except Exception as e:
    raise ImportError(
        "Could not import pymer4. Run this notebook in the environment where pymer4/rpy2/R/lme4 are installed. "
        "Original error: " + repr(e)
    )

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

R callback write-console: Error: package or namespace load failed for 'tibble' in inDL(x, as.logical(local), as.logical(now), ...):
 unable to load shared object 'C:/Users/fiviw/miniforge3/envs/eelbrain/Lib/R/library/stats/libs/x64/stats.dll':
  LoadLibrary failure:  The specified module could not be found.

  
R callback write-console: Error in inDL(x, as.logical(local), as.logical(now), ...) : 
  unable to load shared object 'C:/Users/fiviw/miniforge3/envs/eelbrain/Lib/R/library/stats/libs/x64/stats.dll':
  LoadLibrary failure:  The specified module could not be found.
  


ImportError: Could not import pymer4. Run this notebook in the environment where pymer4/rpy2/R/lme4 are installed. Original error: RRuntimeError("Error in inDL(x, as.logical(local), as.logical(now), ...) : \n  unable to load shared object 'C:/Users/fiviw/miniforge3/envs/eelbrain/Lib/R/library/stats/libs/x64/stats.dll':\n  LoadLibrary failure:  The specified module could not be found.\n")

In [3]:
# --------------------------------------------------
# 1. Path setup
# --------------------------------------------------

# The notebook is intended to live in statistics1/.
# This path logic also works if you run it from the project root.

CWD = Path.cwd()

if CWD.name == "statistics1":
    STATISTICS_DIR = CWD
elif (CWD / "statistics1").exists():
    STATISTICS_DIR = CWD / "statistics1"
else:
    # Fallback: assume current directory is the statistics directory
    STATISTICS_DIR = CWD

ENVELOPE_PATH = STATISTICS_DIR / "aad_trial_level_results.csv"
ENV_ONSET_PATH = STATISTICS_DIR / "envelope_onsets" / "aad_trial_level_results_env_onset.csv"

ACCURACY_REPORT_PATH = STATISTICS_DIR / "pipeline_accuracy_glmm_report.csv"
CORRELATION_REPORT_PATH = STATISTICS_DIR / "pipeline_correlation_lmm_report.csv"

print("Statistics directory:", STATISTICS_DIR)
print("Envelope file:", ENVELOPE_PATH)
print("Envelope + onsets file:", ENV_ONSET_PATH)

Statistics directory: c:\Projects\AAD_Fagprojekt_v2\statistics1
Envelope file: c:\Projects\AAD_Fagprojekt_v2\statistics1\aad_trial_level_results.csv
Envelope + onsets file: c:\Projects\AAD_Fagprojekt_v2\statistics1\envelope_onsets\aad_trial_level_results_env_onset.csv


In [4]:
# --------------------------------------------------
# 2. Load data
# --------------------------------------------------

env = pd.read_csv(ENVELOPE_PATH)
env_onset = pd.read_csv(ENV_ONSET_PATH)

print("Envelope shape:", env.shape)
print("Envelope + onsets shape:", env_onset.shape)

display(env.head())
display(env_onset.head())

Envelope shape: (1408, 13)
Envelope + onsets shape: (1408, 31)


,subject,hearing_status,group_HI,scalp_only,trial_index,r_att,r_ign,r_diff,correct,subject_decoding_accuracy,subject_n_trials,subject_n_correct,summary_path
0,sub-001,hi,1,True,0,0.108249,0.046398,0.061851,1,0.96875,32,31,results_baseline_all\sub-001\sub-001_backward_...
1,sub-001,hi,1,True,1,0.154476,0.016525,0.137951,1,0.96875,32,31,results_baseline_all\sub-001\sub-001_backward_...
2,sub-001,hi,1,True,2,0.071035,-0.007117,0.078152,1,0.96875,32,31,results_baseline_all\sub-001\sub-001_backward_...
3,sub-001,hi,1,True,3,0.117353,0.083574,0.033779,1,0.96875,32,31,results_baseline_all\sub-001\sub-001_backward_...
4,sub-001,hi,1,True,4,0.070426,-0.034507,0.104933,1,0.96875,32,31,results_baseline_all\sub-001\sub-001_backward_...


,subject,hearing_status,group_HI,trial_index,correct,score,r_att_envelope,r_ign_envelope,diff_envelope,r_att_onset,...,basis_window,test,partitions,error,selective_stopping,scale_data,score_mode,feature_weights,summary_path,trial_csv_path
0,sub-001,NaN,NaN,0,1,0.075024,0.106912,0.045254,0.061658,0.133078,...,hamming,1,NaN,l2,1,True,mean,"1.0,1.0",results_env_onset\sub-001\sub-001_backward_mtr...,results_env_onset\sub-001\sub-001_backward_mtr...
1,sub-001,NaN,NaN,1,1,0.129550,0.155028,0.015747,0.139281,0.177719,...,hamming,1,NaN,l2,1,True,mean,"1.0,1.0",results_env_onset\sub-001\sub-001_backward_mtr...,results_env_onset\sub-001\sub-001_backward_mtr...
2,sub-001,NaN,NaN,2,1,0.100022,0.070399,-0.008114,0.078513,0.083431,...,hamming,1,NaN,l2,1,True,mean,"1.0,1.0",results_env_onset\sub-001\sub-001_backward_mtr...,results_env_onset\sub-001\sub-001_backward_mtr...
3,sub-001,NaN,NaN,3,0,-0.018967,0.117675,0.083960,0.033715,0.048953,...,hamming,1,NaN,l2,1,True,mean,"1.0,1.0",results_env_onset\sub-001\sub-001_backward_mtr...,results_env_onset\sub-001\sub-001_backward_mtr...
4,sub-001,NaN,NaN,4,1,0.123232,0.068591,-0.031645,0.100237,0.112976,...,hamming,1,NaN,l2,1,True,mean,"1.0,1.0",results_env_onset\sub-001\sub-001_backward_mtr...,results_env_onset\sub-001\sub-001_backward_mtr...


In [ ]:
# --------------------------------------------------
# 3. Helper functions
# --------------------------------------------------

def group_label_from_group_hi(series: pd.Series) -> pd.Series:
    """Convert group_HI coding to readable group labels."""
    return np.where(series.astype(int) == 1, "HI", "NH")


def fisher_z(r: pd.Series | np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Fisher-z transform with clipping to avoid infinities at exactly +/-1."""
    r = np.asarray(r, dtype=float)
    r = np.clip(r, -1 + eps, 1 - eps)
    return np.arctanh(r)


def clean_pymer4_coefs(coefs: pd.DataFrame, model_type: str) -> pd.DataFrame:
    """Make pymer4 coefficient tables easier to report and save."""
    out = coefs.copy()
    out = out.reset_index().rename(columns={"index": "term"})

    # Normalize common pymer4 column names across versions
    rename_map = {
        "Estimate": "estimate",
        "Est.": "estimate",
        "SE": "se",
        "Std.Err": "se",
        "Z-stat": "z",
        "T-stat": "t",
        "P-val": "p_value",
        "Pr(>|z|)": "p_value",
        "Pr(>|t|)": "p_value",
        "2.5_ci": "ci_low",
        "97.5_ci": "ci_high",
        "CI_lower": "ci_low",
        "CI_upper": "ci_high",
    }
    out = out.rename(columns={c: rename_map.get(c, c) for c in out.columns})

    # Keep useful columns if present
    preferred = ["term", "estimate", "se", "z", "t", "p_value", "ci_low", "ci_high"]
    existing = [c for c in preferred if c in out.columns]
    out = out[existing].copy()

    if model_type == "glmm":
        # Odds ratios are meaningful for logistic models
        if "estimate" in out.columns:
            out["odds_ratio"] = np.exp(out["estimate"])
        if {"ci_low", "ci_high"}.issubset(out.columns):
            out["ci_low_odds_ratio"] = np.exp(out["ci_low"])
            out["ci_high_odds_ratio"] = np.exp(out["ci_high"])

    # Add readable p-value formatting
    if "p_value" in out.columns:
        out["p_value_formatted"] = out["p_value"].apply(
            lambda p: "< .001" if pd.notna(p) and p < 0.001 else (f"{p:.4f}" if pd.notna(p) else "")
        )

    return out

In [ ]:
# --------------------------------------------------
# 4. Prepare long-format accuracy data
# --------------------------------------------------

acc_env = env[["subject", "group_HI", "trial_index", "correct"]].copy()
acc_env["pipeline"] = "envelope"

acc_env_onset = env_onset[["subject", "group_HI", "trial_index", "correct"]].copy()
acc_env_onset["pipeline"] = "envelope_onsets"

acc_df = pd.concat([acc_env, acc_env_onset], ignore_index=True)

acc_df["group"] = group_label_from_group_hi(acc_df["group_HI"])
acc_df["correct"] = acc_df["correct"].astype(int)

# Set reference levels through categorical ordering:
# - pipeline reference: envelope
# - group reference: NH
acc_df["pipeline"] = pd.Categorical(acc_df["pipeline"], categories=["envelope", "envelope_onsets"], ordered=False)
acc_df["group"] = pd.Categorical(acc_df["group"], categories=["NH", "HI"], ordered=False)
acc_df["subject"] = acc_df["subject"].astype(str)
acc_df["trial_index"] = acc_df["trial_index"].astype(int)

print("Accuracy long-format shape:", acc_df.shape)
display(acc_df.groupby(["pipeline", "group"])["correct"].agg(["mean", "sum", "count"]))

In [ ]:
# --------------------------------------------------
# 5. Accuracy GLMM
# --------------------------------------------------

# Main fixed effect: pipeline
# Secondary fixed effect: group
# Random effect: subject

accuracy_formula = "correct ~ pipeline + group + (1|subject)"

accuracy_model = glmer(
    accuracy_formula,
    data=acc_df,
    family="binomial"
)

accuracy_fit = accuracy_model.fit()
print(accuracy_fit)

accuracy_report = clean_pymer4_coefs(accuracy_model.coefs, model_type="glmm")
accuracy_report.to_csv(ACCURACY_REPORT_PATH, index=False)

print("Saved accuracy report to:", ACCURACY_REPORT_PATH)
display(accuracy_report)

In [5]:
# --------------------------------------------------
# 6. Prepare correlation delta data
# --------------------------------------------------

# Envelope-only:
# delta_z = z(r_att) - z(r_ign)

corr_env = env[["subject", "group_HI", "trial_index", "r_att", "r_ign"]].copy()
corr_env["pipeline"] = "envelope"
corr_env["z_att"] = fisher_z(corr_env["r_att"])
corr_env["z_ign"] = fisher_z(corr_env["r_ign"])
corr_env["delta_z"] = corr_env["z_att"] - corr_env["z_ign"]

# Envelope + onsets:
# We Fisher-z transform the envelope and onset correlations separately, then average them.
# This avoids averaging raw Pearson correlations before transformation.

corr_env_onset = env_onset[
    [
        "subject",
        "group_HI",
        "trial_index",
        "r_att_envelope",
        "r_ign_envelope",
        "r_att_onset",
        "r_ign_onset",
    ]
].copy()

corr_env_onset["pipeline"] = "envelope_onsets"

corr_env_onset["z_att"] = np.nanmean(
    np.column_stack([
        fisher_z(corr_env_onset["r_att_envelope"]),
        fisher_z(corr_env_onset["r_att_onset"]),
    ]),
    axis=1
)

corr_env_onset["z_ign"] = np.nanmean(
    np.column_stack([
        fisher_z(corr_env_onset["r_ign_envelope"]),
        fisher_z(corr_env_onset["r_ign_onset"]),
    ]),
    axis=1
)

corr_env_onset["delta_z"] = corr_env_onset["z_att"] - corr_env_onset["z_ign"]

corr_df = pd.concat(
    [
        corr_env[["subject", "group_HI", "trial_index", "pipeline", "z_att", "z_ign", "delta_z"]],
        corr_env_onset[["subject", "group_HI", "trial_index", "pipeline", "z_att", "z_ign", "delta_z"]],
    ],
    ignore_index=True
)

corr_df["group"] = group_label_from_group_hi(corr_df["group_HI"])
corr_df["pipeline"] = pd.Categorical(corr_df["pipeline"], categories=["envelope", "envelope_onsets"], ordered=False)
corr_df["group"] = pd.Categorical(corr_df["group"], categories=["NH", "HI"], ordered=False)
corr_df["subject"] = corr_df["subject"].astype(str)
corr_df["trial_index"] = corr_df["trial_index"].astype(int)

print("Correlation long-format shape:", corr_df.shape)
display(corr_df.groupby(["pipeline", "group"])["delta_z"].agg(["mean", "std", "count"]))

NameError: name 'fisher_z' is not defined

In [ ]:
# --------------------------------------------------
# 7. Correlation LMM
# --------------------------------------------------

# Outcome:
# delta_z = Fisher-z attended correlation - Fisher-z ignored correlation
#
# Main fixed effect: pipeline
# Secondary fixed effect: group
# Random effect: subject

correlation_formula = "delta_z ~ pipeline + group + (1|subject)"

correlation_model = lmer(
    correlation_formula,
    data=corr_df
)

correlation_fit = correlation_model.fit()
print(correlation_fit)

correlation_report = clean_pymer4_coefs(correlation_model.coefs, model_type="lmm")
correlation_report.to_csv(CORRELATION_REPORT_PATH, index=False)

print("Saved correlation report to:", CORRELATION_REPORT_PATH)
display(correlation_report)

## Does the pipeline effect differ between NH and HI

The models have the framework:

- `pipeline` = main fixed effect
- `group` = secondary fixed effect
- `subject` = random intercept

Interaction models for testing whether the onsets affect the two hearing groups differently:

```python
correct ~ pipeline * group + (1|subject)
delta_z ~ pipeline * group + (1|subject)
```

The interaction term answers whether adding onsets changes performance differently for HI and NH listeners.

In [ ]:
# --------------------------------------------------
# 8. Interaction models:
# Does the pipeline effect differ between NH and HI?
# --------------------------------------------------

# These models include:
# - main effect of pipeline
# - main effect of group
# - pipeline × group interaction
# - random subject intercept

ACCURACY_INTERACTION_REPORT_PATH = STATISTICS_DIR / "pipeline_accuracy_glmm_interaction_report.csv"
CORRELATION_INTERACTION_REPORT_PATH = STATISTICS_DIR / "pipeline_correlation_lmm_interaction_report.csv"


# --------------------------------------------------
# 8.1 Accuracy GLMM with pipeline × group interaction
# --------------------------------------------------

accuracy_interaction_formula = "correct ~ pipeline * group + (1|subject)"

accuracy_interaction_model = glmer(
    accuracy_interaction_formula,
    data=acc_df,
    family="binomial"
)

accuracy_interaction_fit = accuracy_interaction_model.fit()
print(accuracy_interaction_fit)

accuracy_interaction_report = clean_pymer4_coefs(
    accuracy_interaction_model.coefs,
    model_type="glmm"
)

accuracy_interaction_report.to_csv(
    ACCURACY_INTERACTION_REPORT_PATH,
    index=False
)

print("Saved accuracy interaction report to:", ACCURACY_INTERACTION_REPORT_PATH)
display(accuracy_interaction_report)